In [1]:
import numpy as np
import pandas as pd
import random
from functools import partial

from collections import defaultdict
from sklearn import metrics

import src.load_data as ld
import src.set_analysis_func as func
import src.func_optimized as func_opt


In [2]:
# load embedding
node_vectors = np.loadtxt(
    './data/embedding/node2vec_consensus.csv', delimiter=',')
node_list = []
with open('./data/embedding/consensus_node.txt', 'r') as f:
    for line in f:
        node_list.append(line.strip())
        
S = metrics.pairwise.cosine_similarity(node_vectors, node_vectors)

In [3]:
# create gene to embedding id mapping
g_node2index = {j:i for i,j in enumerate(node_list)}
g_index2node = {i:j for i,j in enumerate(node_list)}
g_node2index = defaultdict(lambda:-1, g_node2index)

In [4]:
# load gene set data
GO_data = ld.load_gmt(
    './data/gene_sets/hsa_experimental_eval_BP_propagated.gmt')

GO2indices = ld.term2indexes(
    GO_data, g_node2index, upper=300, lower=10)

In [5]:
# generate background gene list
GO_all_genes = set()
for x in GO_data:
    GO_all_genes = GO_all_genes.union(GO_data[x])
    
GO_all_genes = GO_all_genes.intersection(node_list)
GO_all_indices = [g_node2index[x] for x in GO_all_genes]

In [13]:
def sample_term_pairs(term_list, n_pairs, seed=0):
    """
    Sample n_pairs random ordered term pairs (t1, t2) with t1 != t2
    from term_list, with a fixed seed for reproducibility.
    """
    rng = random.Random(seed)
    pairs = []
    m = len(term_list)
    for _ in range(n_pairs):
        t1, t2 = rng.sample(term_list, 2)
        pairs.append((t1, t2))
    return pairs

GO_terms = list(GO2indices.keys())
term_pairs = sample_term_pairs(GO_terms, n_pairs=100, seed=42)
len(term_pairs), term_pairs[0]

# build size buckets for GO terms
term_sizes = {t: len(idxs) for t, idxs in GO2indices.items()}

small_terms  = [t for t, s in term_sizes.items() if 10 <= s < 30]
medium_terms = [t for t, s in term_sizes.items() if 30 <= s < 80]
large_terms  = [t for t, s in term_sizes.items() if 80 <= s <= 300]

def sample_pairs_from_bucket(terms, n_pairs, seed=0):
    rng = random.Random(seed)
    pairs = []
    for _ in range(n_pairs):
        t1, t2 = rng.sample(terms, 2)
        pairs.append((t1, t2))
    return pairs

buckets = {
    "small":  sample_pairs_from_bucket(small_terms,  50, seed=1),
    "medium": sample_pairs_from_bucket(medium_terms, 50, seed=2),
    "large":  sample_pairs_from_bucket(large_terms,  50, seed=3),
}


In [14]:
import time
import statistics as stats

def benchmark_andes_callable(andes_callable, term_pairs, warmup=5):
    """
    andes_callable: something like f(terms) -> (true_score, z_score)
    term_pairs: list of (term1, term2)
    warmup: number of calls to ignore for warmup
    """
    # warmup to trigger any lazy imports / cache effects
    for t in term_pairs[:warmup]:
        _ = andes_callable(t)

    times = []

    for t in term_pairs:
        start = time.perf_counter()
        _ = andes_callable(t)
        end = time.perf_counter()
        times.append(end - start)

    total = sum(times)
    avg = total / len(times)
    median = stats.median(times)

    return {
        "n_calls": len(term_pairs),
        "total_seconds": total,
        "avg_seconds": avg,
        "median_seconds": median,
    }

Bucket: small
  old avg: 0.015988396648317575
  new avg: 0.0043474250799044965
  speedup: 3.677670426621545
Bucket: medium
  old avg: 0.048776540823746474
  new avg: 0.008342170719988645
  speedup: 5.846984251577732
Bucket: large
  old avg: 0.2802142099989578
  new avg: 0.02435544665204361
  speedup: 11.50519692790958


In [8]:
def warmup_numba():
    """
    Call this once at startup to trigger numba compilation.
    Avoids compilation overhead on first real call.
    """
    from src.func_optimized import (
        _compute_bma_from_indices,
        _andes_core_cached,
        _andes_background_parallel,
    )
    # Small dummy problem
    dummy_matrix = np.random.randn(50, 50).astype(np.float64)
    dummy_idx = np.arange(10, dtype=np.int64)
    dummy_pop = np.arange(50, dtype=np.int64)

    # Trigger helpers
    _compute_bma_from_indices(dummy_matrix, dummy_idx, dummy_idx)

    # Build tiny cached-style random index arrays for warmup
    ite = 5
    dummy_rand1 = np.stack([dummy_idx for _ in range(ite)], axis=0)
    dummy_rand2 = np.stack([dummy_idx for _ in range(ite)], axis=0)
    dummy_rand1 = np.ascontiguousarray(dummy_rand1, dtype=np.int64)
    dummy_rand2 = np.ascontiguousarray(dummy_rand2, dtype=np.int64)

    _andes_core_cached(dummy_matrix, dummy_idx, dummy_idx, dummy_rand1, dummy_rand2)

    # If you also use the older background kernels, you can still warm them:
    _andes_background_parallel(dummy_matrix, 10, 10, dummy_pop, dummy_pop, 10, 42)

    print("Numba compilation complete.")


In [9]:
warmup_numba()

Numba compilation complete.


In [10]:

f_old = partial(
    func.andes,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_new = partial(
    func_opt.andes,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)


f_cached = partial(
    func_opt.andes_cached,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_np_cached = partial(
    func_opt.andes_cached_np,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
)

f_numba = partial(
    func_opt.andes_numba_parallel,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
    seed=123,      # so it’s reproducible
)

f_final = partial(
    func_opt.andes_numba_cached,
    matrix=S,
    g1_term2index=GO2indices,
    g2_term2index=GO2indices,
    g1_population=GO_all_indices,
    g2_population=GO_all_indices,
    seed=123,      # so it’s reproducible
)

In [11]:
results_old = benchmark_andes_callable(f_old, term_pairs)
results_new = benchmark_andes_callable(f_new, term_pairs)
results_cached = benchmark_andes_callable(f_cached, term_pairs)
# results_np_cached = benchmark_andes_callable(f_np_cached, term_pairs)
results_numba = benchmark_andes_callable(f_numba, term_pairs)
results_final = benchmark_andes_callable(f_final, term_pairs)

In [12]:
results_old, results_new, results_cached, results_numba, results_final

({'n_calls': 100,
  'total_seconds': 4.816596949938685,
  'avg_seconds': 0.04816596949938685,
  'median_seconds': 0.03357956244144589},
 {'n_calls': 100,
  'total_seconds': 4.0758475384209305,
  'avg_seconds': 0.040758475384209306,
  'median_seconds': 0.0262955414946191},
 {'n_calls': 100,
  'total_seconds': 4.032571291434579,
  'avg_seconds': 0.040325712914345786,
  'median_seconds': 0.025574021507054567},
 {'n_calls': 100,
  'total_seconds': 1.7742339177057147,
  'avg_seconds': 0.017742339177057146,
  'median_seconds': 0.01712468749610707},
 {'n_calls': 100,
  'total_seconds': 0.8601891227299348,
  'avg_seconds': 0.008601891227299348,
  'median_seconds': 0.007757770537864417})

In [ ]:
def run_bucket_benchmark(name, pairs, f_old, f_final):
    res_old = benchmark_andes_callable(f_old, pairs)
    res_new = benchmark_andes_callable(f_final, pairs)
    print(f"Bucket: {name}")
    print("  old avg:",  res_old["avg_seconds"])
    print("  new avg:",  res_new["avg_seconds"])
    print("  speedup:",  res_old["avg_seconds"] / res_new["avg_seconds"])
    return res_old, res_new

bucket_results = {}
for name, pairs in buckets.items():
    bucket_results[name] = run_bucket_benchmark(name, pairs, f_old, f_final)


In [ ]:
def benchmark_vs_ite(terms, f_builder, ites=(100, 300, 1000, 2000)):
    """
    f_builder(ite) -> callable that runs andes with that ite.
    """
    import time
    results = []
    for it in ites:
        f = f_builder(it)
        # warmup
        _ = f(terms)
        start = time.perf_counter()
        _ = f(terms)
        end = time.perf_counter()
        elapsed = end - start
        results.append((it, elapsed))
        print(f"ite={it}: {elapsed:.4f} s")
    return results


In [ ]:
test_pair = term_pairs[0]  # or choose a medium-large set manually

def build_old(ite):
    return partial(
        func.andes,
        matrix=S,
        g1_term2index=GO2indices,
        g2_term2index=GO2indices,
        g1_population=GO_all_indices,
        g2_population=GO_all_indices,
        ite=ite,
    )

def build_final(ite):
    return partial(
        func_opt.andes_numba_cached,   # or whatever your final is
        matrix=S,
        g1_term2index=GO2indices,
        g2_term2index=GO2indices,
        g1_population=GO_all_indices,
        g2_population=GO_all_indices,
        ite=ite,
        seed=123,
    )

print("Old vs ite:")
old_vs_ite   = benchmark_vs_ite(test_pair, build_old)

print("\nNew vs ite:")
new_vs_ite   = benchmark_vs_ite(test_pair, build_final)


In [ ]:
def compare_results(f_old, f_new, term_pairs):
    diffs_true = []
    diffs_z = []
    for t in term_pairs:
        (t_old, z_old) = f_old(t)
        (t_new, z_new) = f_new(t)
        diffs_true.append(abs(t_old - t_new))
        diffs_z.append(abs(z_old - z_new))
    return np.array(diffs_true), np.array(diffs_z)

# use the same term_pairs you already sampled (or more)
d_true, d_z = compare_results(f_old, f_final, term_pairs)

print("true_score abs diff: max =", d_true.max(), "median =", np.median(d_true))
print("z_score    abs diff: max =", d_z.max(),    "median =", np.median(d_z))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
plt.hist(d_z, bins=30)
plt.title("Absolute z-score differences: old vs final")
plt.xlabel("|Δz|")
plt.ylabel("count")
plt.show()
